## GOAL OF THIS NOTEBOOK

The AI agent should intelligently decide:

| Situation | What Agent Does |
| :--- | :--- |
| baggage issue | use baggage tool |
| flight delay | use compensation tool |
| general airline complaint | use RAG |
| flight status question | use flight status tool |
| unknown question | answer normally |

In [18]:
# Imports
import json
import os
import re

import chromadb
import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer

In [19]:
# Load Environment Variables
load_dotenv("../.env")

OPENROUTER_API_KEY = os.getenv(
    "OPENROUTER_API_KEY"
)

In [20]:
# Initialize LLM
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

MODEL = "openai/gpt-oss-120b:free"

print("LLM Ready!")


LLM Ready!


In [21]:
# Load Embedding Model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [22]:
#Load ChromaDB
chroma_client = chromadb.PersistentClient(
    path="../vector_db"
)

collection = chroma_client.get_collection(
    name="airline_support_knowledge"
)

print("Vector database connected!")

Vector database connected!


In [23]:
# Airline Functions
def check_baggage_policy(airline):

    policies = {
        "Delta": "Delta allows 1 checked bag up to 23kg.",
        "United": "United allows one cabin bag.",
        "American": "American Airlines baggage fees may apply."
    }

    return policies.get(
        airline,
        "Policy not found."
    )


def calculate_compensation(delay_hours):

    if delay_hours >= 5:
        return (
            "Passenger eligible for hotel, meals, "
            "and compensation."
        )

    elif delay_hours >= 3:
        return (
            "Passenger eligible for meal vouchers."
        )

    else:
        return "No compensation available."


def check_flight_status(flight_number):

    database = {
        "AI202": "Delayed by 2 hours",
        "DL404": "Boarding",
        "UA101": "Cancelled"
    }

    return database.get(
        flight_number,
        "Flight not found."
    )

In [24]:
# RAG Search Function
def search_airline_knowledge(
    query,
    top_k=3
):

    query_embedding = embedding_model.encode(
        query
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    documents = results["documents"][0]

    return "\n\n".join(documents)

In [25]:
# Tool Definitions
tools = [

    {
        "type": "function",
        "function": {
            "name": "check_baggage_policy",
            "description": "Get baggage policy",
            "parameters": {
                "type": "object",
                "properties": {
                    "airline": {
                        "type": "string"
                    }
                },
                "required": ["airline"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "calculate_compensation",
            "description": "Calculate compensation",
            "parameters": {
                "type": "object",
                "properties": {
                    "delay_hours": {
                        "type": "integer"
                    }
                },
                "required": ["delay_hours"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "check_flight_status",
            "description": "Check flight status",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {
                        "type": "string"
                    }
                },
                "required": ["flight_number"]
            }
        }
    }
]

In [28]:
def airline_agent(user_query):

    response = client.chat.completions.create(
        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": """
                You are an intelligent airline
                customer support AI agent.

                Use tools whenever required.

                Use airline knowledge base
                for customer complaint understanding.
                """
            },

            {
                "role": "user",
                "content": user_query
            }
        ],

        tools=tools
    )

    message = response.choices[0].message

    # Detect Tool Calls
    if message.tool_calls:

        tool_call = message.tool_calls[0]

        tool_name = tool_call.function.name

        tool_args = json.loads(
            tool_call.function.arguments
        )

        # Execute Tools
        if tool_name == "calculate_compensation":

            result = calculate_compensation(
                tool_args["delay_hours"]
            )

        elif tool_name == "check_baggage_policy":

            result = check_baggage_policy(
                tool_args["airline"]
            )

        elif tool_name == "check_flight_status":

            result = check_flight_status(
                tool_args["flight_number"]
            )

        else:
            result = "Tool not found."

        # Final AI Response
        final_response = client.chat.completions.create(

            model=MODEL,

            messages=[
                {
                    "role": "user",
                    "content": user_query
                },

                message.model_dump(),

                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                }
            ]
        )

        return final_response.choices[0].message.content

    # RAG Fallback
    else:

        rag_result = search_airline_knowledge(
            user_query
        )

        final_prompt = f"""
        Customer Question:
        {user_query}

        Airline Knowledge:
        {rag_result}

        Give a helpful airline support response.
        """

        final_response = client.chat.completions.create(
            model=MODEL,

            messages=[
                {
                    "role": "user",
                    "content": final_prompt
                }
            ]
        )

        return final_response.choices[0].message.content

In [29]:
# Test The Agent
response = airline_agent(
    "My flight got delayed by 6 hours"
)

print(response)

I’m sorry to hear about the delay—six hours is certainly inconvenient. Below is a quick rundown of what you’re typically entitled to under EU Regulation 261/2004 (which many other jurisdictions model after), as well as some practical steps you can take right away.

---

## 1. What your rights usually are

| Delay length | Flight distance* | Compensation (EU 261) | Additional assistance |
|--------------|------------------|----------------------|------------------------|
| **≥ 3 h** | Up to 1 500 km | **€250** | Meals & refreshments, two free calls/SMS, **if overnight → hotel + transport** |
| **≥ 3 h** | 1 500–3 500 km | **€400** | Same as above |
| **≥ 4 h** | > 3 500 km | **€600** | Same as above |

\*Distance is measured between the scheduled departure and arrival airports.

Because your delay is **6 hours**, you fall into the “≥ 3 h” bracket for compensation. The exact amount depends on the flight’s scheduled distance:

| Example routes | Approx. distance | Compensation |
|--------